# 08.10 - Transformers Synthesis & Review

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

A cumulative integration unit combining transformer architecture, attention, positional encoding, tokenizers, Hugging Face, and fine-tuning into a complete project.

## 2. Why Does This Matter?

Knowing individual components is not enough. You must build an end-to-end transformer application independently, making design decisions and debugging real issues.

## 3. Prerequisites

- All previous units in Phase 08.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a complete fine-tuning pipeline from scratch.
- Evaluate models with appropriate metrics.
- Perform error analysis and identify improvement directions.
- Make and justify model/hyperparameter choices.

## 5. Mental Model

This is the capstone of Phase 08: combine everything into a sentiment-analysis-style pipeline. We use a tiny transformer on synthetic data to keep it fast and CPU-friendly.

```text
Raw text -> Tokenization -> Train/val/test split -> Fine-tune -> Evaluate -> Error analysis -> Save
```


## 6. Build the Pipeline: Data & Tokenization

Create a synthetic sentiment dataset and tokenize it.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import numpy as np
from transformers import BertConfig, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

torch.manual_seed(42)
np.random.seed(42)

# Synthetic sentiment data: positive words (5-9) vs negative words (10-14)
def make_data(n):
    texts, labels = [], []
    for i in range(n):
        if i % 2 == 0:
            texts.append([5, 6, 7, 8, 9])
            labels.append(1)  # positive
        else:
            texts.append([10, 11, 12, 13, 14])
            labels.append(0)  # negative
    return texts, labels

train_ids, train_labels = make_data(60)
val_ids, val_labels = make_data(20)
test_ids, test_labels = make_data(20)

train_ds = Dataset.from_dict({"input_ids": train_ids, "labels": train_labels})
val_ds = Dataset.from_dict({"input_ids": val_ids, "labels": val_labels})
test_ds = Dataset.from_dict({"input_ids": test_ids, "labels": test_labels})
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 60, Val: 20, Test: 20


## 7. Fine-tune the Model

Use the Trainer API with proper train/val split and evaluation.


In [2]:
config = BertConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=2,
    num_attention_heads=2, intermediate_size=64, num_labels=2,
)
model = BertForSequenceClassification(config)

args = TrainingArguments(
    output_dir="./synthesis_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to=[],
    disable_tqdm=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    return {"accuracy": float(acc)}

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
print("\nFine-tuning complete.")


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.6804', 'eval_accuracy': '1', 'eval_runtime': '0.1413', 'eval_samples_per_second': '141.5', 'eval_steps_per_second': '21.23', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.49it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.26it/s]

...

{'eval_loss': '0.6709', 'eval_accuracy': '1', 'eval_runtime': '0.014', 'eval_samples_per_second': '1432', 'eval_steps_per_second': '214.8', 'epoch': '3'}


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 47.22it/s]

{'train_runtime': '3.681', 'train_samples_per_second': '48.9', 'train_steps_per_second': '6.52', 'train_loss': '0.6789', 'epoch': '3'}

Fine-tuning complete.


## 8. Evaluate on the Held-Out Test Set

Compute accuracy, precision, recall, and F1.


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = test_labels

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1:        1.000

Confusion matrix:
[[10  0]
 [ 0 10]]


## 9. Error Analysis

Examine misclassified examples to find patterns.


In [4]:
misclassified = [(i, y_true[i], y_pred[i]) for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"Misclassified: {len(misclassified)} / {len(y_true)}")
for idx, true_l, pred_l in misclassified[:10]:
    print(f"  Example {idx}: true={true_l}, predicted={pred_l}, tokens={test_ids[idx]}")

if not misclassified:
    print("\nNo misclassifications - the synthetic task is fully separable.")
    print("In a real task, error analysis would reveal patterns like ambiguous or mislabeled examples.")


Misclassified: 0 / 20

No misclassifications - the synthetic task is fully separable.
In a real task, error analysis would reveal patterns like ambiguous or mislabeled examples.


## 10. Save the Model & Tokenizer

Persist the fine-tuned model for later use.


In [5]:
import os
save_dir = "./saved_sentiment_model"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
print(f"Model saved to {save_dir}")
print("Files:", os.listdir(save_dir))

# Reload to verify
reloaded = BertForSequenceClassification.from_pretrained(save_dir)
print("\nModel reloaded successfully.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 29.24it/s]

Model saved to ./saved_sentiment_model
Files: ['config.json', 'model.safetensors']


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 41/41 [00:00<00:00, 3430.96it/s]


Model reloaded successfully.


## 11. Design Decisions & Justification

**Model choice:** We used a tiny BERT-style encoder-only model because:
- Encoder-only models are ideal for classification (bidirectional context).
- A tiny model keeps training fast on CPU.

**Hyperparameters:**
- Learning rate 2e-4 (small, to preserve pretrained knowledge).
- 3 epochs with early stopping via `load_best_model_at_end`.
- Weight decay 0.01 for regularization.

**Evaluation:** Accuracy, precision, recall, F1, and confusion matrix give a complete picture.

## 12. Failure Case: Wrong Tokenizer

If you used the wrong tokenizer, input IDs would not match the model's vocabulary, producing garbage.

## 13. Debugging: Common Errors

- **Overfitting**: LR too high, data too small. Lower LR, freeze layers.
- **Underfitting**: LR too low. Increase LR.
- **Class imbalance**: use weighted loss.
- **Catastrophic forgetting**: smaller LR, freeze more layers.

## 14. Real-World Considerations

- For 100K reviews, use a larger model and more epochs; for 1K, freeze layers.
- Use LoRA for large models to reduce compute.
- Set random seeds for reproducibility.

## 15. Common Mistakes

- Not matching tokenizer to model.
- Not evaluating during training.
- Not saving checkpoints.

## 16. When NOT to Use

- When a simple model (e.g. logistic regression on TF-IDF) suffices.
- When you have very little labeled data.

## 17. Challenge

Add a third class to the synthetic data and re-run the pipeline.


In [6]:
# Challenge: 3-class classification
def make_data3(n):
    texts, labels = [], []
    for i in range(n):
        c = i % 3
        if c == 0:
            texts.append([5, 6, 7])
        elif c == 1:
            texts.append([10, 11, 12])
        else:
            texts.append([15, 16, 17])
        labels.append(c)
    return texts, labels

t3, l3 = make_data3(60)
ds3 = Dataset.from_dict({"input_ids": t3, "labels": l3})
config3 = BertConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=2,
    num_attention_heads=2, intermediate_size=64, num_labels=3,
)
m3 = BertForSequenceClassification(config3)
a3 = TrainingArguments(output_dir="./synth3", num_train_epochs=3, per_device_train_batch_size=8,
                       learning_rate=2e-4, report_to=[], disable_tqdm=True)
tr3 = Trainer(model=m3, args=a3, train_dataset=ds3)
tr3.train()
print("\n3-class fine-tuning complete.")


D:\CODE\complete ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 72.96it/s]

{'train_runtime': '0.4841', 'train_samples_per_second': '371.9', 'train_steps_per_second': '49.58', 'train_loss': '1.092', 'epoch': '3'}

3-class fine-tuning complete.


## 18. Closed-Book Recall

Without looking back:

1. Why did you choose an encoder-only model for classification?
2. What would happen if you used the wrong tokenizer?
3. How would you handle a dataset with 100K reviews vs 1K?
4. What patterns do you look for in misclassified examples?

## 19. Teach-Back Questions

Explain to another person:

- The full fine-tuning pipeline.
- How to evaluate and improve a fine-tuned model.

## 20. Summary

You built a complete sentiment-analysis pipeline: data, tokenization, fine-tuning, evaluation, error analysis, and model saving.

## 21. Further Experiment

- Compare BERT vs DistilBERT vs RoBERTa (requires internet).
- Implement LoRA and compare with full fine-tuning.
- Build a simple inference API with FastAPI.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, transformers, datasets, numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
